# Example 13 — ROOT TTree input with uproot

This example writes a small synthetic ROOT `DecayTree`, loads it through DalitzPlotFitter's uproot interface and uses the resulting `PhaseSpaceSample` as fit input.


In [ ]:
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import uproot

from dalitzplotfitter import (DecayChannel, DecayModel, NonResonant, RealImag, Resonance, Parameter, Minimizer, enable_x64, read_phase_space_sample, weighted_resample)
from dalitzplotfitter.likelihood import UnbinnedNLL
enable_x64()

channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1.0,0.0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.50,0.10))],normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))


## 1. Generate a toy and write it to ROOT


In [ ]:
pool=model.generate_phase_space(120000,seed=13001)
toy=weighted_resample(jax.random.key(13002),pool,pool.weights*model.intensity(pool.as_dict()),12000,replace=True)
root_path=Path('example13_b2kpipi.root')
with uproot.recreate(root_path) as f:
    f['DecayTree']={'S12':np.asarray(toy.s12),'S13':np.asarray(toy.s13),'S23':np.asarray(toy.s23)}
print(root_path.resolve())


## 2. Load ROOT data as a PhaseSpaceSample


In [ ]:
data=read_phase_space_sample(root_path,'DecayTree',s12='S12',s13='S13',s23='S23')
print('events:',data.size)
print('first s13:',float(data.s13[0]))
plt.figure(figsize=(7,5.5))
plt.hist2d(np.asarray(data.s13),np.asarray(data.s23),bins=70)
plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel(r'$s_{23}$ [GeV$^2$]'); plt.title('ROOT-loaded B+ -> K+ pi+ pi- sample'); plt.colorbar(label='events'); plt.show()


## 3. Use the ROOT-loaded sample in the fitter

For clarity this example fits one amplitude coefficient while keeping the remaining model fixed.


In [ ]:
nr_x=Parameter.coefficient('NR.x',-0.30,bounds=(-1.5,0.5),step=0.02,owner='NR')
fit_model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1.0,0.0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(nr_x,0.10),name='NR')],normalization_method='square-dalitz',normalization_resolution=300,normalization_pair=(0,2))
pdf=fit_model.pdf()
nll=UnbinnedNLL(pdf.logpdf,data.as_dict())
result=Minimizer(nll,(nr_x,),verbose=1).fit(start_values={'NR.x':-0.30},simplex=True,ncall=10000)
print('generated NR.x = -0.50')
print('start NR.x     = -0.30')
print('fitted NR.x    =',float(result.values['NR.x']))
print('valid          =',result.valid)


## 4. Projection using the fitted model


In [ ]:
fit_values={'NR.x':float(result.values['NR.x'])}
proj=model.generate_phase_space(100000,seed=13003)
w=np.asarray(proj.weights*fit_model.intensity(proj.as_dict(),fit_values))
bins=np.linspace(float(data.s13.min()),float(data.s13.max()),70)
plt.figure(figsize=(7,5))
plt.hist(np.asarray(data.s13),bins=bins,histtype='step',density=True,label='ROOT data')
plt.hist(np.asarray(proj.s13),bins=bins,weights=w,histtype='step',density=True,label='fitted model')
plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel('normalized entries'); plt.legend(); plt.show()
